In [19]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [ ]:
# Example: Control for Z
X = df[["treatment_X", "confounder_Z"]]  # include Z
y = df["outcome_Y"]
X = sm.add_constant(X)
model = sm.OLS(y, X).fit() # Potentially unbiased



# Example: Do NOT control for Z
X = df[["treatment_X"]]  # Z omitted
y = df["outcome_Y"]
X = sm.add_constant(X)
model = sm.OLS(y, X).fit() # Possibly biased by Z

KeyError: "None of [Index(['treatment_X', 'confounder_Z'], dtype='object')] are in the [columns]"

In [21]:
np.random.seed(42)
n = 1000

#. Z (Confounder) ─▶ X (Treatment) ─▶ Y (Outcome)
#. │                                        ▲
#. └────────────────────────────────────────┘


# Confounder Z: affects both treatment X and outcome Y
Z = np.random.normal(0, 1, n)

# Treatment X: influenced by Z
X = 0.5 * Z + np.random.normal(0, 1, n)

# Outcome Y: influenced by X and Z
Y = 0.7 * X + 0.6 * Z + np.random.normal(0, 1, n)


#.  X (Treatment) ─▶ C (Collider) ◀─ Y (Outcome)

# Collider C: influenced by both X and Y (DO NOT control for this)
C = 0.4 * X + 0.4 * Y + np.random.normal(0, 1, n)

# DataFrame
df = pd.DataFrame({"Z": Z, "X": X, "Y": Y, "C": C})

In [22]:
Z[:5]

array([ 0.49671415, -0.1382643 ,  0.64768854,  1.52302986, -0.23415337])

In [4]:
# Naive regression (no controls)
model_naive = sm.OLS(df["Y"], sm.add_constant(df["X"])).fit()
print("Naive (no controls):")
print(model_naive.summary())

Naive (no controls):
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.442
Model:                            OLS   Adj. R-squared:                  0.441
Method:                 Least Squares   F-statistic:                     789.9
Date:                Tue, 26 Aug 2025   Prob (F-statistic):          1.69e-128
Time:                        19:20:12   Log-Likelihood:                -1541.8
No. Observations:                1000   AIC:                             3088.
Df Residuals:                     998   BIC:                             3097.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0003      0.03

In [5]:
# Proper regression controlling for the confounder Z ✅
model_with_confounder = sm.OLS(df["Y"], sm.add_constant(df[["X", "Z"]])).fit()
print("\nControlling for Confounder Z:")
print(model_with_confounder.summary())


Controlling for Confounder Z:
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.578
Model:                            OLS   Adj. R-squared:                  0.578
Method:                 Least Squares   F-statistic:                     683.9
Date:                Tue, 26 Aug 2025   Prob (F-statistic):          1.03e-187
Time:                        19:20:20   Log-Likelihood:                -1401.5
No. Observations:                1000   AIC:                             2809.
Df Residuals:                     997   BIC:                             2824.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0061

In [6]:
# Improper regression controlling for the collider C ❌
model_with_collider = sm.OLS(df["Y"], sm.add_constant(df[["X", "C"]])).fit()
print("\nControlling for Collider C (BAD):")
print(model_with_collider.summary())


Controlling for Collider C (BAD):
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.541
Model:                            OLS   Adj. R-squared:                  0.540
Method:                 Least Squares   F-statistic:                     587.8
Date:                Tue, 26 Aug 2025   Prob (F-statistic):          2.33e-169
Time:                        19:20:27   Log-Likelihood:                -1443.8
No. Observations:                1000   AIC:                             2894.
Df Residuals:                     997   BIC:                             2908.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.